In [1]:
# External Imports
import sys
from pathlib import Path
import torch
import torch.optim as optim
import torch.nn as nn

# Internal Imports
sys.path.insert(0, '../src')
from src.Dataset.mri_split import split_patients
from src.Dataset.data_loaders import get_dataloaders
from src.Dataset.cache import load_cache

In [2]:
accepted_data = {}
rejected_data = {}
try:
    accepted_data = load_cache(Path("../data/processed/cache/accepted_data.json"), Path.cwd().parent)
    rejected_data = load_cache(Path("../data/processed/cache/rejected_data.json"), Path.cwd().parent)
except BaseException as e:
    print(e)

[INFO]  Cache loaded from ../data/processed/cache/accepted_data.json
[INFO]  Cache loaded from ../data/processed/cache/rejected_data.json


In [3]:
SPLIT_SEED = 42
train_patients, val_patients, test_patients = split_patients(accepted_data, seed=SPLIT_SEED)

[INFO]  Splitting dataset with seed 42...
[INFO]  Train Set: 76  | Tumor Ratio: 0.352
[INFO]  Valid Set: 17  | Tumor Ratio: 0.360
[INFO]  Test Set:  17   | Tumor Ratio: 0.372


In [4]:
BATCH_SIZE = 32
train_dataloader, val_dataloader, test_dataloader = get_dataloaders(
    train_patients,
    val_patients,
    test_patients,
    batch_size=BATCH_SIZE,
    segmentation=True
)

/opt/miniconda3/envs/BrainTumorMRIClassification/lib/python3.13/site-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


In [5]:
images, masks = next(iter(train_dataloader))

In [6]:
print(images.shape)
print(masks.shape)
print(masks.unique())
print(images.dtype)
print(masks.dtype)

torch.Size([32, 3, 224, 224])
torch.Size([32, 1, 224, 224])
tensor([0., 1.])
torch.float32
torch.float32


# CHECKPOINT

Next is writing the training loop